# 🧠 Lista de Exercícios — Cenário 1: Worm na Rede (CSBANK)

## Introdução
Você é um analista de segurança da **CSBANK**.  
Um engenheiro de redes enviou por e-mail um arquivo `.pcap` com indícios de um ataque do tipo **worm**, que se espalha pela rede pela **porta 445 (SMB)**.  

Seu desafio é analisar o tráfego, extrair evidências e descobrir **qual máquina foi o paciente zero** — o primeiro host infectado e responsável pela propagação inicial.

---

## 🎯 Sua missão

### Nível 0 — Preparação Básica
- Baixe o arquivo PCAP disponível em:  
  👉 [https://link.com.br](https://link.com.br)
- Salve-o com o nome `csbank_traffic.pcap`.
- Importe o arquivo para o Python e confirme que foi carregado corretamente.

**Tarefas:**
1. Fazer o download do arquivo (Linux/macOS):
```bash
curl -L -o csbank_traffic.pcap "https://link.com.br"
```
2. Validar o arquivo no ambiente:
```bash
!ls -lh csbank_traffic.pcap
```
3. Criar um novo notebook `Worm_445_Analise.ipynb` e carregar o PCAP.

---

### Nível 1 — Ambiente e Bibliotecas
Prepare seu ambiente com as bibliotecas necessárias para análise.

**Requisitos:**
- `pandas`
- `numpy`
- `scikit-learn`
- `scipy`
- Uma biblioteca de leitura de pacotes (ex.: `scapy`, `pyshark` ou `pypcap`)

**Instalação:**
```bash
pip install pandas numpy scikit-learn scipy scapy pyshark
```

**Leitura inicial do PCAP:**
```python
from scapy.all import rdpcap
pkts = rdpcap("csbank_traffic.pcap")
print(len(pkts), pkts[0].summary())
```

**Desafio:**
- Quantos pacotes TCP existem?
- Quantos pacotes envolvem a porta 445?

Dica: use filtros como `tcp.dport == 445` para contar rapidamente.

### Nível 2 — Exploração e Engenharia de Features
Agora você vai extrair informações relevantes por **IP de origem (src_ip)**.

**Objetivo:** entender o comportamento de cada host da rede e identificar padrões de propagação.

**Features sugeridas:**
- `pkt_count` → total de pacotes enviados
- `byte_count` → total de bytes transmitidos
- `unique_dst` → número de destinos únicos
- `pkt_count_on_target` → pacotes que envolvem a porta 445
- `byte_count_on_target` → bytes trocados via porta 445
- `unique_dst_on_target` → destinos distintos atingidos na porta 445
- `syn_count` / `rst_count` → tentativas e falhas de conexão
- `duration_s` → tempo entre primeiro e último pacote
- `pps` / `pps_target` → pacotes por segundo (geral e na porta 445)
- `fraction_to_target` → proporção de pacotes envolvendo a porta 445

**Atividade:**
1. Criar um DataFrame `hosts_features` com uma linha por IP.
2. Plotar gráficos (ex.: `histogramas` ou `boxplots`) para observar anomalias.
3. Interpretar: quais IPs apresentam comportamento fora do padrão?

### Nível 3 — Modelo de Machine Learning (IsolationForest)
Chegou a hora de aplicar aprendizado de máquina para identificar os IPs mais suspeitos.

**Passos:**
1. Criar a matriz `X` com as features e normalizar com `StandardScaler`.
2. Treinar o modelo:
```python
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(hosts_features.select_dtypes(float))

iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=42)
iso.fit(X_scaled)
scores = -iso.decision_function(X_scaled)
hosts_features["anomaly_score"] = scores
```
3. Ordenar IPs por `anomaly_score` (maior = mais suspeito).
4. Plotar gráfico de barras com os top-10 IPs suspeitos.
5. Calcular *z-scores* das features para entender por que foram considerados anômalos.
6. Determinar o **paciente zero**:
   - IP com **menor timestamp (`first_ts`)**
   - Alto `pkt_count_on_target`
   - Alto `unique_dst_on_target`
   - Alto `anomaly_score`

### 🧩 Entrega final — Relatório técnico
No final do notebook, escreva uma célula em **Markdown** com o título:

> 🧠 Relatório Final — Caso Worm na Rede (CSBANK)

Inclua:
- IP do **paciente zero**
- Timestamp do primeiro tráfego
- Evidências observadas (features + z-scores)
- Gráfico dos top suspeitos
- Interpretação do resultado

---

## 🚀 Desafios Extras (Opcional)
- Implementar janelas temporais (sliding window de 60 segundos) e observar a propagação.
- Enriquecer a análise com parsing SMB usando `pyshark` (detectar comandos SMB “Tree Connect” e “Trans2”).
- Comparar comportamento com um PCAP “limpo” (sem worm).
- Criar um dashboard simples (matplotlib/seaborn) com evolução temporal da propagação.

---

**Objetivo final:** demonstrar domínio sobre análise de tráfego, engenharia de features e uso de machine learning para **detecção de comportamento anômalo em redes**.